In [1]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# 🕵️ Inspecting Detected Entities

Dig into the entity detection pipeline output -- what was detected,
what the LLM validator kept or dropped, and where entities appear in the text.

This notebook is for users who need to debug detection quality,
tune labels and/or thresholds, or investigate downstream replacement or rewriting results.

We use **Annotate** mode because it preserves the original text while tagging each entity
with its label, making it ideal for reviewing detection quality.

> **Privacy warning:** `Annotate` does not anonymize the text. Sensitive values
> remain in the output, so use it only for inspection -- not as a privacy-safe
> production strategy.

#### 📚 What you'll learn

- Run the detection pipeline and inspect its output using Annotate mode
- View tagged text with entities marked inline
- Break down detected entities by label, source, and unique value
- Identify and triage failed records

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
  - The default `build.nvidia.com` (NVIDIA Build) setup is a convenient way to try Anonymizer and iterate on previews. Use of NVIDIA Build is subject to NVIDIA Build's own terms of service and privacy practices, which are separate from and independent of the NeMo Framework library. NVIDIA Build is intended for evaluation and testing purposes only and may not be used in production environments. Do not upload any confidential information or personal data when using NVIDIA Build. Your use of NVIDIA Build is logged for security purposes and to improve NVIDIA products and services.
  - Request and token rate limits on `build.nvidia.com` vary by account and model access, and lower-volume development access can be slow for full-dataset runs. Start with `preview()` on a small sample, then move to your own endpoint for production data and usage.
- Import the core classes -- this notebook uses `Annotate` to keep original values visible.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.
- `configure_logging(LoggingConfig.default())` keeps logs at INFO. Switch to `LoggingConfig.debug()` when troubleshooting.

In [2]:
import getpass
import os
from collections import Counter

import pandas as pd

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [3]:
from anonymizer import Annotate, Anonymizer, AnonymizerConfig, AnonymizerInput, LoggingConfig, configure_logging

configure_logging(LoggingConfig.default())

In [4]:
anonymizer = Anonymizer()

[11:05:50] [INFO] 🔧 Anonymizer initialized with 3 model configs


[11:05:50] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[11:05:50] [INFO]   |-- ✅ validator: gpt-oss-120b


[11:05:50] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 👁️ Preview

- Detection runs as part of any strategy. `Annotate` keeps original text visible
  alongside entity labels -- ideal for debugging.
- `trace_dataframe` exposes every internal pipeline column; that's what we explore below.

In [5]:
config = AnonymizerConfig(replace=Annotate())

input_data = AnonymizerInput(
    source="https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)

result = anonymizer.preview(
    config=config,
    data=input_data,
    num_records=3,
)

[11:05:51] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[11:05:51] [INFO] 🔍 Running entity detection on 3 records


[11:05:51] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[11:06:52] [INFO]   |-- 📋 Detection complete — 78 entities found across 3 records (0 failed) [61.0s]


[11:06:52] [INFO]   |-- labels: first_name=23, organization_name=7, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, religious_belief=2, street_address=2, place_name=1, date_of_birth=1, employment_status=1


[11:06:52] [INFO] 🔄 Running Annotate replacement


[11:06:52] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[11:06:52] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


## 🔍 Inspect

- `display_record()` renders an interactive view with entity highlights.

In [6]:
result.display_record(0)

Original,Label,Replacement
Bobby,first_name,"<Bobby, first_name>"
Watford,last_name,"<Watford, last_name>"
40,age,"<40, age>"
Mexican,race_ethnicity,"<Mexican, race_ethnicity>"
veterinarian,occupation,"<veterinarian, occupation>"
Denver,city,"<Denver, city>"
Colorado,state,"<Colorado, state>"
Jefferson High,organization_name,"<Jefferson High, organization_name>"
DVM,degree,"<DVM, degree>"
University of Colorado Boulder,university,"<University of Colorado Boulder, university>"


## 📋 Columns

- `result.dataframe["final_entities"]` is the stable, public entity output.
- `trace_dataframe` contains internal pipeline columns for deeper debugging;
  those underscore-prefixed columns may change between releases.

In [7]:
trace_df = result.trace_dataframe
final_entities = result.dataframe["final_entities"]
print(f"Records: {len(trace_df)}")
print(f"Columns: {list(trace_df.columns)}")

Records: 3
Columns: ['biography', '_anonymizer_record_id', '_raw_detected_entities', '_seed_entities', '_tag_notation', '_seed_validation_candidates', '_seed_tagged_text', '_validated_entities', '_seed_entities_json', '_initial_tagged_text', '_validated_seed_entities', '_augmented_entities', '_merged_entities', '_merged_tagged_text', '_validation_candidates', '_detected_entities', 'biography_with_spans', 'final_entities', '_entities_by_value', '_replacement_map', 'biography_replaced']


## 🎯 Detected entities

- Final entity list after validation. Each entity has `value`, `label`,
  positions, `score`, and `source` (detector / augmenter / name_split / propagation).

In [8]:
row_idx = 0
raw = final_entities.iloc[row_idx]
entities = raw["entities"] if isinstance(raw, dict) else raw
print(f"Record {row_idx}: {len(entities)} entities detected\n")

entity_df = pd.DataFrame(entities)
if not entity_df.empty:
    cols = [c for c in ["value", "label", "start_position", "end_position", "source"] if c in entity_df.columns]
    print(entity_df[cols].to_string())

Record 0: 22 entities detected

                             value              label  start_position  end_position     source
0                            Bobby         first_name               0             5   detector
1                          Watford          last_name               6            13   detector
2                               40                age              17            19   detector
3                          Mexican     race_ethnicity              29            36   detector
4                     veterinarian         occupation              37            49   detector
5                           Denver               city              60            66   detector
6                         Colorado              state              68            76   detector
7                   Jefferson High  organization_name             180           194  augmenter
8                              DVM             degree             210           213   detector
9   University of 

## 🏷️ Labels

- Entity label distribution across all records -- which types are most common.

In [9]:
label_counts = Counter()
for raw in final_entities:
    entity_list = raw["entities"] if isinstance(raw, dict) else raw
    for entity in entity_list:
        label_counts[entity["label"]] += 1

for label, count in label_counts.most_common():
    print(f"  {label}: {count}")

  first_name: 23
  organization_name: 7
  age: 5
  occupation: 5
  city: 4
  state: 4
  degree: 4
  university: 4
  field_of_study: 4
  last_name: 3
  race_ethnicity: 3
  political_view: 3
  language: 2
  religious_belief: 2
  street_address: 2
  place_name: 1
  date_of_birth: 1
  employment_status: 1


## 📡 Sources

- Where each entity came from in the pipeline:
    - `detector` -- GLiNER NER
    - `augmenter` -- LLM-added (missed by GLiNER)
    - `validator` -- LLM decision step over detector-seed entities (keep/reclass/drop); does not emit a separate source value
    - `name_split` -- derived from splitting full names
    - `propagation` -- expanded from validated entities to all text occurrences

In [10]:
source_counts = Counter()
for raw in final_entities:
    entity_list = raw["entities"] if isinstance(raw, dict) else raw
    for entity in entity_list:
        source_counts[entity.get("source", "unknown")] += 1

for source, count in source_counts.most_common():
    print(f"  {source}: {count}")

  detector: 74
  augmenter: 4


## 📊 By value

- Entities grouped by unique value -- this is what drives consistent replacement
  downstream (same name always maps to the same substitute).

In [11]:
row_idx = 0
raw_bv = trace_df.loc[row_idx, "_entities_by_value"]
by_value = raw_bv["entities_by_value"] if isinstance(raw_bv, dict) else raw_bv
print(f"Record {row_idx}: {len(by_value)} unique entity values\n")

for entry in by_value:
    print(f"  {entry['value']!r} -> labels: {entry['labels']}")

Record 0: 19 unique entity values

  '40' -> labels: ['age']
  'Aria' -> labels: ['first_name']
  'Bobby' -> labels: ['first_name']
  'Christian Democrat' -> labels: ['political_view']
  'Colorado' -> labels: ['state']
  'Colorado Veterinary Clinic' -> labels: ['organization_name']
  'DVM' -> labels: ['degree']
  'Denver' -> labels: ['city']
  'English' -> labels: ['language']
  'Jefferson High' -> labels: ['organization_name']
  'Leo' -> labels: ['first_name']
  'Maya' -> labels: ['first_name']
  'Mexican' -> labels: ['race_ethnicity']
  'Rockies' -> labels: ['place_name']
  'University of Colorado Boulder' -> labels: ['university']
  'VCA Animal Hospital' -> labels: ['organization_name']
  'Watford' -> labels: ['last_name']
  'veterinarian' -> labels: ['occupation']
  'wildlife health' -> labels: ['field_of_study']


## ❌ Failures

- Records dropped during detection (LLM timeout, parse error, etc.).
- Check this to understand data loss in your pipeline.

In [12]:
if result.failed_records:
    for fr in result.failed_records:
        print(f"  record_id={fr.record_id}, step={fr.step}, reason={fr.reason}")
else:
    print("No failed records.")

No failed records.


## 📊 (Optional) Score the detections with an LLM judge

- `evaluate()` is a separate, opt-in step that runs LLM-as-judge metrics on the output.
- This notebook uses Annotate, so only **Detection Validity** runs — it flags entities the detector got wrong (false positives, mislabels, boundary errors). Substitute would also enable Type Fidelity, Relational Consistency, and Attribute Fidelity.

In [13]:
evaluated = anonymizer.evaluate(result)
evaluated.display_record(0)

Original,Label,Replacement
Bobby,first_name,"<Bobby, first_name>"
Watford,last_name,"<Watford, last_name>"
40,age,"<40, age>"
Mexican,race_ethnicity,"<Mexican, race_ethnicity>"
veterinarian,occupation,"<veterinarian, occupation>"
Denver,city,"<Denver, city>"
Colorado,state,"<Colorado, state>"
Jefferson High,organization_name,"<Jefferson High, organization_name>"
DVM,degree,"<DVM, degree>"
University of Colorado Boulder,university,"<University of Colorado Boulder, university>"


## ⏭️ Next steps

- **[🕵️ Your First Anonymization](../01_your_first_anonymization/)** --
  the simplest end-to-end replace workflow if you haven't run it yet.
- **[🎯 Choosing a Replacement Strategy](../03_choosing_a_replacement_strategy/)** --
  compare Redact, Annotate, Hash, and Substitute side-by-side.
- **[✏️ Rewriting Biographies](../04_rewriting_biographies/)** --
  generate privacy-safe paraphrases instead of token-level replacements.